In [20]:
import sys
import os
sys.path += [ f'{os.environ["HOME"]}/.local/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages' ]

# Now we can safely import atlasopenmagic
import atlasopenmagic as atom

In [21]:
import uproot # for reading .root files
import time # to measure time to analyse
import math # for mathematical functions such as square root
import awkward as ak # for handling complex and nested data structures efficiently
import numpy as np # # for numerical calculations such as histogramming
import matplotlib.pyplot as plt # for plotting
from matplotlib.ticker import MaxNLocator,AutoMinorLocator # for minor ticks
from lmfit.models import PolynomialModel, GaussianModel # for the signal and background fits
import vector #to use vectors
import requests # for HTTP access
import aiohttp # HTTP client support
import pandas as pd
import json

In [22]:
atom.set_release('2025e-13tev-beta')

Release '2025e-13tev-beta' already active with cached metadata.
Active release: 2025e-13tev-beta. (Datasets path: REMOTE)


In [23]:
CONFIG_FILE = "config.json"

with open(CONFIG_FILE, "r") as f:
    config = json.load(f)

In [24]:
signal_dsid_list = config["samples"]["signal"]["dsids"]
background_dsid_list = config["samples"]["background"]["dsids"]

signal_nevents = config["samples"]["signal"]["nevents"]
background_nevents = config["samples"]["background"]["nevents"]

variables = config["variables"]

lumi = config["options"]["lumi"]
skim = config["options"]["skim"]
checkpoint_dir = config["options"]["checkpoint_dir"]

In [25]:
def get_xsec_weight(metadata, lumi):
    return (
        lumi * 1000
        * metadata["cross_section_pb"]
        * metadata["genFiltEff"]
        * metadata["kFactor"]
        / metadata["sumOfWeights"]
    )

def get_N_inclusive(metadata, lumi):
    return (
        lumi * 1000
        * metadata["cross_section_pb"]
        * metadata["genFiltEff"]
        * metadata["kFactor"]
    )

def get_inclusive_yield(metadata, lumi):
    return (
        lumi * 1000
        * metadata["cross_section_pb"]
        * metadata["genFiltEff"]
        * metadata["kFactor"]
    )

def calc_weight(xsec_weight, weight_arr, data):
    for variable in weight_arr:
        xsec_weight = xsec_weight * data[variable]
    return xsec_weight

In [26]:
def create_objects(data, variables):

    prefixes = {
        "lep": "lep_",
        "jet": "jet_",
        "tau": "tau_",
        "photon": "photon_",
        "largeRJet": "largeRJet_",
        "ScaleFactor": "ScaleFactor_",
        "trig": "trig_",
        "met": "met_"}

    objects = {}

    for variable in variables:
        # special case: met
        if variable == "met":
            if "met" not in objects:
                objects["met"] = {}
            objects["met"]["met"] = data[variable]
            continue

        found = False

        for category, prefix in prefixes.items():
            if variable.startswith(prefix):
                name = variable[len(prefix):]
                if category not in objects:
                    objects[category] = {}
                objects[category][name] = data[variable]
                found = True
                break

        if not found:
            if "info" not in objects:
                objects["info"] = {}
            objects["info"][variable] = data[variable]

    # zip everything
    for category in objects:
        objects[category] = ak.zip(objects[category], depth_limit=1)

    return objects

In [ ]:
def create_blackbox(signal_dsid_list, background_dsid_list, variables, signal_nevents, background_nevents, lumi=36, 
                    skim="noskim", checkpoint_dir="blackbox_checkpoint",):
    
    print("Variables:", variables)

    # create checkpoint directories
    events_dir = os.path.join(checkpoint_dir, "events")
    completed_file = os.path.join(checkpoint_dir, "completed_dsids.json")

    os.makedirs(events_dir, exist_ok=True)


    # load checkpoint information
    if os.path.exists(completed_file):

        print(f"Load checkpoint: {checkpoint_dir}")

        with open(completed_file, "r") as f:
            completed_dsids = set(json.load(f))

        print(f"{len(completed_dsids)} DSIDs already processed.")
        print(f"Completed DSIDs: {completed_dsids}")

    else:
        print("No checkpoint found. Start from scratch.")
        completed_dsids = set()


    # calculate inclusive yields
    yield_dict = {}

    signal_yield_sum = 0
    background_yield_sum = 0

    for dsid in signal_dsid_list:

        metadata = atom.get_metadata(dsid)
        inc_yield = get_inclusive_yield(metadata, lumi)

        yield_dict[dsid] = inc_yield
        signal_yield_sum += inc_yield


    for dsid in background_dsid_list:

        metadata = atom.get_metadata(dsid)
        inc_yield = get_inclusive_yield(metadata, lumi)

        yield_dict[dsid] = inc_yield
        background_yield_sum += inc_yield


    # calculate number of events per DSID
    nevents_per_sample_dict = {}

    for dsid in signal_dsid_list:
        nevents_per_sample = (signal_nevents / signal_yield_sum * yield_dict[dsid])
        nevents_per_sample_dict[dsid] = nevents_per_sample


    for dsid in background_dsid_list:
        nevents_per_sample = (background_nevents / background_yield_sum * yield_dict[dsid])
        nevents_per_sample_dict[dsid] = nevents_per_sample


    # save completed DSIDs
    def save_completed_dsids():
        with open(completed_file, "w") as f:
            json.dump(sorted(completed_dsids), f, indent=2)
        print(f"Checkpoint information saved: " f"{len(completed_dsids)} completed DSIDs")


    # process samples
    dsid_labels = ([(dsid, 1) for dsid in signal_dsid_list] + [(dsid, 0) for dsid in background_dsid_list])

    for dsid, label in dsid_labels:

        dsid_str = str(dsid)

        print("\n" + "=" * 60)
        print(f"Current DSID: {dsid}")
        print(f"Label: {label}")
        print(f"Already processed: {dsid_str in completed_dsids}")

        # skip already completed DSID
        if dsid_str in completed_dsids:
            print(f"DSID {dsid} already processed -> SKIP")
            continue

        # number of events to collect
        target_events = int(nevents_per_sample_dict[dsid])

        remaining = target_events
        collected_events = 0

        print(f"Target events: {target_events}")

        # temporary storage for this DSID
        chunks = {}
        label_chunks = []

        # get ROOT files
        file_list = atom.get_urls(dsid, skim, protocol="root", cache=False)
        print("Number of files:", len(file_list))

        # loop over ROOT files
        for file_number, afile in enumerate(file_list, start=1):

            print(f"\nProcessing file " f"{file_number}/{len(file_list)}")

            if collected_events >= target_events:
                break

            # read ROOT file chunk-by-chunk
            for data in uproot.iterate(afile + ":analysis", variables, library="ak", step_size=100_000,):

                if len(data) == 0:
                    continue

                # only take events that are still needed
                if len(data) > remaining:
                    data = data[:remaining]

                n_events = len(data)

                # create objects from variable names
                new_objects = create_objects(data, variables)
                label_chunks.append(ak.Array([label] * n_events))

                # save in chunks
                for name, obj in new_objects.items():
                    if name not in chunks:
                        chunks[name] = []
                    chunks[name].append(obj)

                # update counters
                collected_events += n_events
                remaining = target_events - collected_events

                print(f"Collected events: " f"{collected_events}/{target_events}")

                if collected_events >= target_events:
                    break

        # check whether enough events were found
        if collected_events < target_events:
            print(f"WARNING: Only found " f"{collected_events}/{target_events} events " f"for DSID {dsid}.")

        # combine chunks for this DSID
        dsid_arrays = {}

        for name, chunk_list in chunks.items():
            if len(chunk_list) > 0:
                dsid_arrays[name] = ak.concatenate(chunk_list, axis=0)

        # combine labels
        if len(label_chunks) == 0:
            print(f"WARNING: No events collected for DSID {dsid} -> SKIP")
            continue
        dsid_arrays["label"] = ak.concatenate(label_chunks, axis=0)

        # create one Awkward record for this DSID
        dsid_data = ak.zip(dsid_arrays, depth_limit=1)

        # save DSID checkpoint
        checkpoint_file = os.path.join(events_dir, f"dsid_{dsid_str}.parquet")

        print(f"Saving {len(dsid_data)} events to " f"{checkpoint_file}")

        ak.to_parquet(dsid_data, checkpoint_file)

        # mark DSID as completed
        completed_dsids.add(dsid_str)

        save_completed_dsids()


    # load all checkpoint files
    print("\nLoading all checkpoint files...")

    parquet_files = [os.path.join(events_dir, f) for f in os.listdir(events_dir) if f.endswith(".parquet")]

    parquet_files.sort()

    if len(parquet_files) == 0:
        raise RuntimeError("No events found in checkpoint.")

    for parquet_file in parquet_files:
        ds = ak.from_parquet(parquet_file)

    # load and concatenate
    data = ak.from_parquet(parquet_files)

    print(f"Loaded {len(data)} events " f"from {len(parquet_files)} parquet files.")


    # shuffle
    rng = np.random.default_rng(25)
    indices = rng.permutation(len(data))
    data = data[indices]

    # labels
    labels = data["label"]

    return data, labels

In [28]:
data, labels = create_blackbox(signal_dsid_list, background_dsid_list, variables, signal_nevents, background_nevents, skim=skim, lumi=lumi, checkpoint_dir=checkpoint_dir)

Variables: ['lep_pt', 'lep_eta', 'lep_phi', 'lep_e', 'lep_n', 'jet_pt', 'jet_eta', 'jet_phi', 'jet_e', 'jet_n', 'tau_pt', 'tau_eta', 'tau_phi', 'tau_e', 'tau_n', 'photon_pt', 'photon_eta', 'photon_phi', 'photon_e', 'photon_n', 'largeRJet_pt', 'largeRJet_eta', 'largeRJet_phi', 'largeRJet_e', 'ScaleFactor_ElTRIGGER', 'ScaleFactor_FTAG', 'trigP', 'met']
No checkpoint found. Start from scratch.

Current DSID: 301209
Label: 1
Already processed: False
Target events: 10
Number of files: 1

Processing file 1/1
Collected events: 10/10
Saving 10 events to blackbox_checkpoint/events/dsid_301209.parquet
Checkpoint information saved: 1 completed DSIDs

Current DSID: 700323
Label: 0
Already processed: False
Target events: 1
Number of files: 1

Processing file 1/1
Collected events: 1/1
Saving 1 events to blackbox_checkpoint/events/dsid_700323.parquet
Checkpoint information saved: 2 completed DSIDs

Current DSID: 700324
Label: 0
Already processed: False
Target events: 6
Number of files: 1

Processing 

In [31]:
data.lep

<Array [{pt: [15.9, ...], eta: ..., ...}, ...] type='107 * {pt: var * float...'>

In [ ]:
labels

<Array [0, 0, 0, 0, 0, 1, 0, 0, ..., 1, 0, 0, 0, 0, 0, 0, 0] type='107 * int64'>